# 使用 Langchain、Qdrant 和 OpenAI 实现问答

本 notebook 介绍如何使用 Langchain、作为知识库的 Qdrant 以及 OpenAI 嵌入来实现一个问答系统。如果你不熟悉 Qdrant，建议先阅读 [Getting_started_with_Qdrant_and_OpenAI.ipynb](Getting_started_with_Qdrant_and_OpenAI.ipynb) notebook。

本 notebook 展示了一个端到端的流程：
1. 使用 OpenAI API 计算嵌入。
2. 将嵌入存储到本地 Qdrant 实例中，构建知识库。
3. 使用 OpenAI API 将原始文本查询转换为嵌入。
4. 使用 Qdrant 在已创建的集合中执行最近邻搜索，以找到相关上下文。
5. 让 LLM 在给定上下文中找出答案。

所有步骤都将简化为调用相应的 Langchain 方法。

## 前提条件

为完成本练习，我们需要准备以下几样东西：

1. Qdrant 服务器实例。在本例中是一个本地 Docker 容器。
2. 用于与向量数据库交互的 [qdrant-client](https://github.com/qdrant/qdrant_client) 库。
3. 作为框架的 [Langchain](https://github.com/hwchase17/langchain)。
3. 一个 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)。

### 启动 Qdrant 服务器

我们将使用运行在 Docker 容器中的本地 Qdrant 实例。最简单的启动方式是使用附带的 [docker-compose.yaml] 文件并运行以下命令：

In [1]:
! docker-compose up -d

Starting qdrant_qdrant_1 ... 
ting qdrant_qdrant_1 ... done

我们可以运行一个简单的 curl 命令来验证服务器是否成功启动：

In [2]:
! curl http://localhost:6333

{"title":"qdrant - vector search engine","version":"1.0.1"}

### 安装依赖

本 notebook 显然需要 `openai`、`langchain` 和 `qdrant-client` 这几个包。


In [ ]:
! pip install openai qdrant-client "langchain==0.0.100" wget

### 准备 OpenAI API 密钥

OpenAI API 密钥用于对文档和查询进行向量化。

如果你还没有 OpenAI API 密钥，可以从 [https://beta.openai.com/account/api-keys](https://beta.openai.com/account/api-keys) 获取。

获取密钥后，请运行以下命令，将其添加到环境变量 `OPENAI_API_KEY` 中：

In [ ]:
! export OPENAI_API_KEY="your API key"

In [4]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.
import os

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

if os.getenv("OPENAI_API_KEY") is not None:
    print("OPENAI_API_KEY is ready")
else:
    print("OPENAI_API_KEY environment variable not found")

OPENAI_API_KEY is ready


## 加载数据

在本节中，我们将加载一些自然问题及其答案的数据。所有数据都将用于构建一个以 Qdrant 为知识库的 Langchain 应用。

In [5]:
import wget

# All the examples come from https://ai.google.com/research/NaturalQuestions
# This is a sample of the training set that we download and extract for some
# further processing.
wget.download("https://storage.googleapis.com/dataset-natural-questions/questions.json")
wget.download("https://storage.googleapis.com/dataset-natural-questions/answers.json")

100% [..............................................................................] 95372 / 95372

'answers.json'

In [6]:
import json

with open("questions.json", "r") as fp:
    questions = json.load(fp)

with open("answers.json", "r") as fp:
    answers = json.load(fp)

In [7]:
print(questions[0])

when is the last episode of season 8 of the walking dead


In [8]:
print(answers[0])

No . overall No. in season Title Directed by Written by Original air date U.S. viewers ( millions ) 100 `` Mercy '' Greg Nicotero Scott M. Gimple October 22 , 2017 ( 2017 - 10 - 22 ) 11.44 Rick , Maggie , and Ezekiel rally their communities together to take down Negan . Gregory attempts to have the Hilltop residents side with Negan , but they all firmly stand behind Maggie . The group attacks the Sanctuary , taking down its fences and flooding the compound with walkers . With the Sanctuary defaced , everyone leaves except Gabriel , who reluctantly stays to save Gregory , but is left behind when Gregory abandons him . Surrounded by walkers , Gabriel hides in a trailer , where he is trapped inside with Negan . 101 `` The Damned '' Rosemary Rodriguez Matthew Negrete & Channing Powell October 29 , 2017 ( 2017 - 10 - 29 ) 8.92 Rick 's forces split into separate parties to attack several of the Saviors ' outposts , during which many members of the group are killed ; Eric is critically injure

## 定义链

Langchain 已经与 Qdrant 集成，会为给定的文档列表完成所有索引工作。在本例中，我们将存储已有的答案集合。

In [9]:
from langchain.vectorstores import Qdrant
from langchain.embeddings import OpenAIEmbeddings
from langchain import VectorDBQA, OpenAI

embeddings = OpenAIEmbeddings()
doc_store = Qdrant.from_texts(
    answers, embeddings, host="localhost" 
)

此时所有可能的答案都已存储在 Qdrant 中，因此我们可以定义完整的 QA 链了。

In [10]:
llm = OpenAI()
qa = VectorDBQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    vectorstore=doc_store,
    return_source_documents=False,
)

## 搜索数据

数据存入 Qdrant 后，我们就可以开始提问了。问题会被 OpenAI 模型自动向量化，生成的向量将用于在 Qdrant 中查找可能匹配的答案。检索到之后，最相似的答案会被整合进发送给 OpenAI 大语言模型的提示词中。各服务之间的通信如下图所示：

![](https://qdrant.tech/articles_data/langchain-integration/flow-diagram.png)


In [11]:
import random

random.seed(52)
selected_questions = random.choices(questions, k=5)

In [12]:
for question in selected_questions:
    print(">", question)
    print(qa.run(question), end="\n\n")

> where do frankenstein and the monster first meet
 Victor and the Creature first meet in the mountains.

> who are the actors in fast and furious
 The actors in the Fast and Furious films are Vin Diesel, Paul Walker, Michelle Rodriguez, Jordana Brewster, Tyrese Gibson, Ludacris, Lucas Black, Sung Kang, Gal Gadot, Dwayne Johnson, Matt Schulze, Chad Lindberg, Johnny Strong, Eva Mendes, Devon Aoki, Nathalie Kelley, Bow Wow, Tego Calderón, Don Omar, Elsa Pataky, Kurt Russell, Nathalie Emmanuel, Scott Eastwood, Noel Gugliemi, Ja Rule, Thom Barry, Ted Levine, Minka Kelly, James Remar, Amaury Nolasco, Michael Ealy, MC Jin, Brian Goodman, Lynda Boyd, Jason Tobin, Neela, Liza Lapira, Alimi Ballard, Yorgo Constantine, Geoff Meed, Jeimy Osorio, Max William Crane, Charlie & Miller Kimsey, Eden Estrella, Romeo Santos, John Brotherton, Helen Mirren, Celestino Cornielle, Janmarco Santiago, Carlos De La Hoz, James Ayoub, Rick Yune, Cole Hauser, Brian Tee, John Ortiz, Luke Evans, Jason Statham, Charli

### 自定义提示词模板

Langchain 中的 `stuff` 链类型使用一个包含问题和上下文文档的特定提示词。默认提示词如下所示：

```text
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
{context}
Question: {question}
Helpful Answer:
```

不过，我们可以提供自己的提示词模板来改变 OpenAI LLM 的行为，同时仍使用 `stuff` 链类型。重要的是要保留 `{context}` 和 `{question}` 这两个占位符。

#### 尝试自定义提示词

我们可以尝试使用不同的提示词模板，让模型：
1. 如果知道答案，就用一句话作答。
2. 如果不知道问题的答案，就推荐一个随机的歌曲名。

In [13]:
from langchain.prompts import PromptTemplate

In [14]:
custom_prompt = """
Use the following pieces of context to answer the question at the end. Please provide
a short single-sentence summary answer only. If you don't know the answer or if it's 
not present in given context, don't try to make up an answer, but suggest me a random 
unrelated song title I could listen to. 
Context: {context}
Question: {question}
Helpful Answer:
"""

In [15]:
custom_prompt_template = PromptTemplate(
    template=custom_prompt, input_variables=["context", "question"]
)

In [16]:
custom_qa = VectorDBQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    vectorstore=doc_store,
    return_source_documents=False,
    chain_type_kwargs={"prompt": custom_prompt_template},
)

In [17]:
random.seed(41)
for question in random.choices(questions, k=5):
    print(">", question)
    print(custom_qa.run(question), end="\n\n")

> what was uncle jesse's original last name on full house
Uncle Jesse's original last name on Full House was Cochran.

> when did the volcano erupt in indonesia 2018
No volcanic eruption is mentioned in the given context. Suggested Song: "Ring of Fire" by Johnny Cash.

> what does a dualist way of thinking mean
Dualist way of thinking means that the mind and body are separate entities, with the mind being a non-physical substance.

> the first civil service commission in india was set up on the basis of recommendation of
The first Civil Service Commission in India was not set up on the basis of a recommendation.

> how old do you have to be to get a tattoo in utah
In Utah, you must be at least 18 years old to get a tattoo.

